# ROCLING 2026 DSA — E10 / E11 / E12（multi-seed + 多 encoder + ensemble）

專跑 **拿分主線**，不需要 L2/L3 的大 pkl。流程：
1. E10：MacBERT + L1 跑 5 個 seed
2. E11：RoBERTa-wwm-ext 與 large 各一顆
3. E12：把所有 run 用 dimension-wise weighted / mean 融合出 submission

## 這是「VS Code + Colab 擴充套件」版
- 程式碼在 **Colab 遠端機器**上跑，你本機 `data/` 那台機器看不到 → cell 2 改成在 runtime 上 **git clone**。
- **先確認 GPU**：右上角 Select Kernel → Colab → 選一個 **premium GPU** runtime（Pro 才有 A100 / L4）。
- **cell 1 會印出你實際連到哪顆 GPU**。若拿到 A100/L4（VRAM ≥ 24GB），可把下面各 cell 的
  `--batch_size` 調大（如 macbert 64、roberta-large 16）加速；若只有 T4（16GB）就用預設值。</cell>


In [1]:
# 1) 安裝套件 + 確認「我連到的是哪顆 Colab GPU」
!pip -q install "transformers>=4.40" jieba scikit-learn scipy

import torch, subprocess
print('=' * 60)
if not torch.cuda.is_available():
    print('⚠️  沒有 GPU！請右上角 Select Kernel → Colab → 選 premium GPU runtime')
else:
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'✅ GPU：{name}')
    print(f'   VRAM：{vram:.1f} GB')
    print(f'   CUDA capability：{torch.cuda.get_device_capability(0)}')
    # 對照：T4=16GB / V100=16GB / L4=24GB / A100=40或80GB
    tier = ('A100（最強，Pro premium）' if 'A100' in name else
            'L4（Pro premium）'         if 'L4'   in name else
            'V100'                       if 'V100' in name else
            'T4（免費層常見）'           if 'T4'   in name else '其他')
    print(f'   → 判定：{tier}')
    if vram >= 24:
        print('   💡 VRAM 夠大，可把下面 --batch_size 調大加速（macbert 64 / roberta-large 16）')
print('=' * 60)
# 完整 nvidia-smi（型號、記憶體、driver 版本）
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)

✅ GPU：NVIDIA A100-SXM4-40GB
   VRAM：39.5 GB
   CUDA capability：(8, 0)
   → 判定：A100（最強，Pro premium）
   💡 VRAM 夠大，可把下面 --batch_size 調大加速（macbert 64 / roberta-large 16）
Thu Jul  9 01:57:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P0             47W /  400W |       6MiB /  40960MiB

In [2]:
# 2) 從 GitHub 拉程式到 Colab runtime（取代 upload+解壓；VS Code 模式本機檔案不在遠端機器上）
import os
BRANCH = 'feat/ensemble-teacher-student-experiments'
REPO = 'https://<REDACTED_TOKEN>@github.com/chen0427ok/DSA-NIFT.git'   # 若 repo 為 private：改成 https://<TOKEN>@github.com/chen0427ok/DSA-NIFT.git

if not os.path.exists('repo'):
    !git clone -q -b {BRANCH} {REPO} repo
os.chdir('/content/repo' if os.path.exists('/content/repo') else 'repo')
os.makedirs('outputs/preds', exist_ok=True)
print('工作目錄:', os.getcwd())
print('data:', sorted(os.listdir('data')) if os.path.isdir('data') else '❌ 找不到 data/')
assert os.path.exists('data/train.csv') and os.path.exists('train_v2.py'), '程式或資料沒拉到，檢查 branch/repo 名稱或 token'
print('✅ 程式與資料就緒')

工作目錄: /content/repo
data: ['dev.csv', 'orign_train_data.csv', 'train.csv', 'val_unlabeled.csv']
✅ 程式與資料就緒


## E10 — Multi-seed ensemble（MacBERT + L1 × 5 seeds）
兩篇得獎論文共識的最大槓桿：同架構跑多個 seed，平掉方差、穩定 arousal 排序。

In [3]:
# 3) E10：5 個 seed（每顆存 outputs/macbert_s{seed}_best.pt 與 preds/）
for s in [42, 1, 2, 3, 4]:
    print(f'\n===== seed {s} =====')
    !python train_v2.py --seed {s} --run_name macbert_s{s} --epochs 4 --batch_size 64


===== seed 42 =====
/usr/local/lib/python3.12/dist-packages/jieba/__init__.py:44: SyntaxWarning: invalid escape sequence '\.'
  re_han_default = re.compile("([\u4E00-\u9FD5a-zA-Z0-9+#&\._%\-]+)", re.U)
/usr/local/lib/python3.12/dist-packages/jieba/__init__.py:46: SyntaxWarning: invalid escape sequence '\s'
  re_skip_default = re.compile("(\r\n|\s)", re.U)
/usr/local/lib/python3.12/dist-packages/jieba/finalseg/__init__.py:78: SyntaxWarning: invalid escape sequence '\.'
  re_skip = re.compile("([a-zA-Z0-9]+(?:\.\d+)?%?)")
run=macbert_s42 | device=cuda | model=hfl/chinese-macbert-base | lex=l1 | aw=1.0 pcc_w=0.0
config.json: 100% 659/659 [00:00<00:00, 4.28MB/s]
tokenizer_config.json: 100% 19.0/19.0 [00:00<00:00, 144kB/s]
vocab.txt: 100% 110k/110k [00:00<00:00, 90.7MB/s]
tokenizer.json: 100% 269k/269k [00:00<00:00, 125MB/s]
added_tokens.json: 100% 2.00/2.00 [00:00<00:00, 12.6kB/s]
special_tokens_map.json: 100% 112/112 [00:00<00:00, 658kB/s]
train=9435 dev=253 val=200
Building prefix dict 

In [4]:
# 4) E10 融合（等權平均；multi-seed 用 mean 最穩，不易 overfit 253 筆 dev）
!python ensemble.py macbert_s42 macbert_s1 macbert_s2 macbert_s3 macbert_s4 \
    --mode mean --name e10_seed_ens

=== 各模型 dev 表現與權重 (mean) ===
macbert_s42              valence_MAE=0.4716 valence_PCC=0.8189 arousal_MAE=0.8241 arousal_PCC=0.6146  | w_V=0.200 w_A=0.200
macbert_s1               valence_MAE=0.4905 valence_PCC=0.8129 arousal_MAE=0.8390 arousal_PCC=0.6154  | w_V=0.200 w_A=0.200
macbert_s2               valence_MAE=0.4813 valence_PCC=0.8113 arousal_MAE=0.8301 arousal_PCC=0.6085  | w_V=0.200 w_A=0.200
macbert_s3               valence_MAE=0.4922 valence_PCC=0.8127 arousal_MAE=0.8516 arousal_PCC=0.6042  | w_V=0.200 w_A=0.200
macbert_s4               valence_MAE=0.4764 valence_PCC=0.8176 arousal_MAE=0.8447 arousal_PCC=0.6038  | w_V=0.200 w_A=0.200
=== Ensemble dev ===
  valence_MAE=0.4720 valence_PCC=0.8229 arousal_MAE=0.8255 arousal_PCC=0.6157
submission -> /content/repo/outputs/e10_seed_ens_submission.csv


## E11 — RoBERTa-wwm-ext + L1
換 encoder bias。large 版對長反思文本可能更會排 arousal，但 T4 記憶體吃緊 → **batch 8 + lr 1e-5**。

In [5]:
# 5) E11a：RoBERTa-wwm-ext（base）
!python train_v2.py --model hfl/chinese-roberta-wwm-ext --run_name roberta_s42 --epochs 4 --batch_size 64

run=roberta_s42 | device=cuda | model=hfl/chinese-roberta-wwm-ext | lex=l1 | aw=1.0 pcc_w=0.0
config.json: 100% 689/689 [00:00<00:00, 4.45MB/s]
tokenizer_config.json: 100% 19.0/19.0 [00:00<00:00, 128kB/s]
vocab.txt: 100% 110k/110k [00:00<00:00, 43.6MB/s]
tokenizer.json: 100% 269k/269k [00:00<00:00, 122MB/s]
added_tokens.json: 100% 2.00/2.00 [00:00<00:00, 15.6kB/s]
special_tokens_map.json: 100% 112/112 [00:00<00:00, 799kB/s]
train=9435 dev=253 val=200
Building prefix dict from the default dictionary ...
Loading model from cache /tmp/jieba.cache
Loading model cost 0.830 seconds.
Prefix dict has been built successfully.
lexicon dim = 10
pytorch_model.bin: 100% 412M/412M [00:04<00:00, 88.4MB/s]
Loading weights: 100% 199/199 [00:00<00:00, 19229.73it/s]
[transformers] BertModel LOAD REPORT from: hfl/chinese-roberta-wwm-ext
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UN

In [6]:
# 6) E11b：RoBERTa-wwm-ext-large（T4 一定要 batch 8 + lr 1e-5，否則 OOM）
!python train_v2.py --model hfl/chinese-roberta-wwm-ext-large \
    --run_name robertaL_s42 --epochs 4 --batch_size 16 --lr 1e-5

run=robertaL_s42 | device=cuda | model=hfl/chinese-roberta-wwm-ext-large | lex=l1 | aw=1.0 pcc_w=0.0
config.json: 100% 690/690 [00:00<00:00, 3.72MB/s]
tokenizer_config.json: 100% 19.0/19.0 [00:00<00:00, 120kB/s]
vocab.txt: 100% 110k/110k [00:00<00:00, 80.2MB/s]
tokenizer.json: 100% 269k/269k [00:00<00:00, 112MB/s]
added_tokens.json: 100% 2.00/2.00 [00:00<00:00, 13.0kB/s]
special_tokens_map.json: 100% 112/112 [00:00<00:00, 780kB/s]
train=9435 dev=253 val=200
Building prefix dict from the default dictionary ...
Loading model from cache /tmp/jieba.cache
Loading model cost 0.858 seconds.
Prefix dict has been built successfully.
lexicon dim = 10
pytorch_model.bin: 100% 1.31G/1.31G [00:07<00:00, 180MB/s]
Loading weights: 100% 391/391 [00:00<00:00, 29989.99it/s]
[transformers] BertModel LOAD REPORT from: hfl/chinese-roberta-wwm-ext-large
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias             

## E12 — 跨 encoder dimension-wise weighted ensemble
valence / arousal **分開**算權重（`score = max(PCC,0)/(MAE+eps)`），arousal 維度只讓會排 arousal 的模型投票。

> dev 只有 253 筆，weighted 可能 overfit → 下面同時跑 mean 版比較；兩者 dev 差距大時**選保守的 mean**。

In [7]:
# 7) E12：把手上所有 run 丟進去（weighted）
!python ensemble.py macbert_s42 macbert_s1 macbert_s2 macbert_s3 macbert_s4 \
    roberta_s42 robertaL_s42 --mode weighted --name e12_enc_ens

print('\n----- 對照：mean 版 -----')
!python ensemble.py macbert_s42 macbert_s1 macbert_s2 macbert_s3 macbert_s4 \
    roberta_s42 robertaL_s42 --mode mean --name e12_enc_mean

=== 各模型 dev 表現與權重 (weighted) ===
macbert_s42              valence_MAE=0.4716 valence_PCC=0.8189 arousal_MAE=0.8241 arousal_PCC=0.6146  | w_V=0.145 w_A=0.148
macbert_s1               valence_MAE=0.4905 valence_PCC=0.8129 arousal_MAE=0.8390 arousal_PCC=0.6154  | w_V=0.138 w_A=0.146
macbert_s2               valence_MAE=0.4813 valence_PCC=0.8113 arousal_MAE=0.8301 arousal_PCC=0.6085  | w_V=0.141 w_A=0.146
macbert_s3               valence_MAE=0.4922 valence_PCC=0.8127 arousal_MAE=0.8516 arousal_PCC=0.6042  | w_V=0.138 w_A=0.141
macbert_s4               valence_MAE=0.4764 valence_PCC=0.8176 arousal_MAE=0.8447 arousal_PCC=0.6038  | w_V=0.143 w_A=0.142
roberta_s42              valence_MAE=0.4608 valence_PCC=0.8279 arousal_MAE=0.8375 arousal_PCC=0.6047  | w_V=0.150 w_A=0.143
robertaL_s42             valence_MAE=0.4605 valence_PCC=0.8039 arousal_MAE=0.8655 arousal_PCC=0.5835  | w_V=0.146 w_A=0.134
=== Ensemble dev ===
  valence_MAE=0.4578 valence_PCC=0.8291 arousal_MAE=0.8268 arousal_PCC=0.6153


## 下載結果
`*_submission.csv` 是官方格式（ID,Valence,Arousal）。`outputs/preds/` 裡的統一預測檔留著，
之後要做 E13 偽標（teacher 用這些 `*_best.pt`）或 E17 校準都會用到。

In [8]:
# 8) 打包結果（VS Code 模式：檔案在 Colab 遠端機器上）
import shutil, os
zip_path = shutil.make_archive('e10_e12_results', 'zip', 'outputs')  # 含所有 submission / preds / best_model 權重
print('已打包 ->', os.path.abspath(zip_path))

# 方式 A：Colab 網頁版才有的下載彈窗（VS Code 模式通常無效，故用 try 包住）
try:
    from google.colab import files
    files.download('e10_e12_results.zip')
except Exception as e:
    print('（VS Code 模式無彈窗下載，改用下面方式取回）:', type(e).__name__)
    print('  方式 B：VS Code 左側「執行階段檔案總管 / Jupyter」找到 repo/e10_e12_results.zip 右鍵 Download')
    print('  方式 C：掛雲端硬碟後複製過去 →')
    print("         from google.colab import drive; drive.mount('/content/drive')")
    print("         shutil.copy('e10_e12_results.zip', '/content/drive/MyDrive/')")

已打包 -> /content/repo/e10_e12_results.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [9]:
# 9) （可選）只看各 submission 的預測分布，快速 sanity check
import pandas as pd, glob
for p in sorted(glob.glob('outputs/*_submission.csv')):
    df = pd.read_csv(p)
    print(f"{os.path.basename(p):32s} V {df.Valence.mean():.2f}±{df.Valence.std():.2f}  "
          f"A {df.Arousal.mean():.2f}±{df.Arousal.std():.2f}")

e10_seed_ens_submission.csv      V 5.63±1.40  A 5.04±0.73
e12_enc_ens_submission.csv       V 5.61±1.37  A 5.04±0.71
e12_enc_mean_submission.csv      V 5.61±1.37  A 5.04±0.71
macbert_s1_submission.csv        V 5.54±1.41  A 5.11±0.77
macbert_s2_submission.csv        V 5.74±1.36  A 5.03±0.73
macbert_s3_submission.csv        V 5.66±1.43  A 5.09±0.76
macbert_s42_submission.csv       V 5.60±1.40  A 5.00±0.77
macbert_s4_submission.csv        V 5.61±1.42  A 4.98±0.71
robertaL_s42_submission.csv      V 5.43±1.23  A 5.03±0.67
roberta_s42_submission.csv       V 5.68±1.40  A 5.02±0.74


In [11]:
  # 把整包結果複製到 Google Drive
  import os, shutil
  from google.colab import drive
  drive.mount('/content/drive')   # 點一下授權你的 Google 帳號

  src = '/content/repo/e10_e12_results.zip'
  dst = '/content/drive/MyDrive/e10_e12_results.zip'
  shutil.copy(src, dst)
  print(f'✅ 已複製到 Drive：{dst}  ({os.path.getsize(dst)/1024**3:.2f} GB)')

Mounted at /content/drive
✅ 已複製到 Drive：/content/drive/MyDrive/e10_e12_results.zip  (3.25 GB)


In [13]:
import os
z_vm = '/content/repo/e10_e12_results.zip'
z_dr = '/content/drive/MyDrive/e10_e12_results.zip'
print('1. Drive 有掛上嗎 :', os.path.exists('/content/drive/MyDrive'))
print('2. zip 在 VM 上嗎  :', os.path.exists(z_vm),
    f'({os.path.getsize(z_vm)/1024**3:.2f} GB)' if os.path.exists(z_vm) else '')
print('3. zip 進 Drive 了嗎:', os.path.exists(z_dr))
if os.path.exists('/content/drive/MyDrive'):
    print('4. MyDrive 根目錄:', sorted(os.listdir('/content/drive/MyDrive'))[:20])

1. Drive 有掛上嗎 : True
2. zip 在 VM 上嗎  : True (3.25 GB)
3. zip 進 Drive 了嗎: True
4. MyDrive 根目錄: ['(日期)｜(姓名)｜練習檔  的副本 的副本 的副本.gdoc', '0719 sanchong', '1 11.gdoc', '1 12.gdoc', '1.gdoc', '1002 固定收益證卷.gdoc', '1140515蔡淑華三人陳報(分割遺產)(修)(1).doc', '123.gdoc', '123未命名文件.gdoc', '2008033020485324.pdf', '2008033120505028.pdf', '2012033100082456.pdf', '2013032915485580.pdf', '2013111323560444.pdf', '37陳𦤶希 308 - 110-1 作文二.docx', '37陳𦤶希 308 - 110-1 作文二.pdf', '5D6E90BB-883B-42A0-A22E-73D146CBB46A.jpeg', '6B9B0B51-25C9-452B-B2DF-7FF0789AB697.MOV', 'A_長春集團_希希不嘻嘻1.MOV', 'A_長春集團_希希不嘻嘻2.MOV']


In [15]:
import os
os.chdir('/content/repo')
!git config user.email "chenbrian930427@gmail.com"
!git config user.name "chen0427ok"
# outputs/ 被 gitignore → 用 -f 強制加；只加小的 csv，不加權重
!git add -f outputs/*_submission.csv outputs/preds/*.csv
print('--- 即將 commit 這些檔 ---')
!git status --short
!git commit -m "E10/E11/E12 結果：submission 與 dev/val 預測檔"
!git push origin feat/ensemble-teacher-student-experiments

--- 即將 commit 這些檔 ---
A  outputs/e10_seed_ens_submission.csv
A  outputs/e12_enc_ens_submission.csv
A  outputs/e12_enc_mean_submission.csv
A  outputs/macbert_s1_submission.csv
A  outputs/macbert_s2_submission.csv
A  outputs/macbert_s3_submission.csv
A  outputs/macbert_s42_submission.csv
A  outputs/macbert_s4_submission.csv
A  outputs/preds/e10_seed_ens_dev.csv
A  outputs/preds/e10_seed_ens_val.csv
A  outputs/preds/e12_enc_ens_dev.csv
A  outputs/preds/e12_enc_ens_val.csv
A  outputs/preds/e12_enc_mean_dev.csv
A  outputs/preds/e12_enc_mean_val.csv
A  outputs/preds/macbert_s1_dev.csv
A  outputs/preds/macbert_s1_val.csv
A  outputs/preds/macbert_s2_dev.csv
A  outputs/preds/macbert_s2_val.csv
A  outputs/preds/macbert_s3_dev.csv
A  outputs/preds/macbert_s3_val.csv
A  outputs/preds/macbert_s42_dev.csv
A  outputs/preds/macbert_s42_val.csv
A  outputs/preds/macbert_s4_dev.csv
A  outputs/preds/macbert_s4_val.csv
A  outputs/preds/robertaL_s42_dev.csv
A  outputs/preds/robertaL_s42_val.csv
A  outputs/p